In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Data Splitting

In this section we'll split the `DB_final.csv` to 3 groups (Google, Gemini, and Marian) Then we will translate each group using the corresponding model and save them to separate files for further evaluation.

In [ ]:
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv('../old_data/DB_final.csv')

le = LabelEncoder()                                        
y_encoded = le.fit_transform(df['Writer'])

_,test_data,_, _= train_test_split(
    df, 
    y_encoded, 
    test_size=0.20, 
    random_state=999, 
    stratify=y_encoded   # same class distribution in train and test
)


In [ ]:
test_data=test_data[["Text","Writer"]].copy()

In [ ]:
test_data["Writer"].value_counts()

In [ ]:
TEXT_LEN=test_data['Text'].apply(len)

In [ ]:
print("Sum :", TEXT_LEN.sum())
print("Desc :\n",TEXT_LEN.describe())

In [ ]:
def IQR(lst):
    q1 = lst.quantile(0.25)
    q3 = lst.quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    return lower_bound, upper_bound

print("IQR bounds:", IQR(TEXT_LEN))
print("number of high outliers:", sum(TEXT_LEN > IQR(TEXT_LEN)[1]))
print("number of low outliers:", sum(TEXT_LEN < IQR(TEXT_LEN)[0]))

In [ ]:
lower_bound, upper_bound = IQR(TEXT_LEN)
clean_test_df = test_data[(TEXT_LEN >= lower_bound) & (TEXT_LEN <= upper_bound)].copy()

_, temp_df = train_test_split(clean_test_df, test_size=0.66, random_state=999,stratify=clean_test_df['Writer'])
_, df_Nllb = train_test_split(temp_df, test_size=0.5, random_state=42,stratify=temp_df['Writer'])

df_Nllb.to_csv('old_data/paraphrasing/nllb_test_data.csv', index=False)

# Nllb

In [ ]:
df_Nllb_TEXT_LEN=df_Nllb['Text'].apply(len)
print("Nllb Sum :", df_Nllb_TEXT_LEN.sum())
print("Nllb Desc :\n",df_Nllb_TEXT_LEN.describe())

In [ ]:
import os
import gc
import torch
import pandas as pd
from tqdm import tqdm
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, pipeline, BitsAndBytesConfig


# VRAM optimization
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
input_file = 'old_data/paraphrasing/nllb_test_data.csv'
output_file = "old_data/paraphrasing/nllb/DB_test_paraphrased_1.3B_ULTRA.csv"
os.makedirs(os.path.dirname(output_file), exist_ok=True)

def clear_vram():
    gc.collect()
    torch.cuda.empty_cache()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForSeq2SeqLM.from_pretrained("facebook/nllb-200-distilled-1.3B", quantization_config=bnb_config, device_map="auto")
tokenizer = AutoTokenizer.from_pretrained("facebook/nllb-200-distilled-1.3B")

# 16 bit batch processing pipeline
translator = pipeline(
    "translation", 
    model=model, 
    tokenizer=tokenizer, 
    batch_size=16  
)

def process_ultra(df):
    if os.path.exists(output_file):
        processed_df = pd.read_csv(output_file)
        start_idx = len(processed_df)
        print(f"Resuming from index: {start_idx}")
    else:
        processed_df = pd.DataFrame()
        start_idx = 0

    remaining_df = df.iloc[start_idx:].copy()
    if remaining_df.empty: return processed_df

    all_chunks = [] # Text chunks for batch processing
    chunk_map = []  #Original indices mapping

    print("Fragmenting texts into chunks...")
    for idx, row in remaining_df.iterrows():
        words = str(row['Text']).split()
        parts = [' '.join(words[i:i + 250]) for i in range(0, len(words), 250)]
        all_chunks.extend(parts)
        chunk_map.extend([idx] * len(parts))

    print(f"Processing {len(all_chunks)} fragments on GPU...")
    
    # ENG -> FRA
    fr_results = []
    for out in tqdm(translator(all_chunks, src_lang="eng_Latn", tgt_lang="fra_Latn", max_length=512, truncation=True), 
                    total=len(all_chunks), desc="Step 1/2: ENG -> FRA"):
        fr_results.append(out['translation_text'])
    
    clear_vram()

    # FRA -> ENG
    en_results = []
    for out in tqdm(translator(fr_results, src_lang="fra_Latn", tgt_lang="eng_Latn", max_length=512, truncation=True), 
                    total=len(fr_results), desc="Step 2/2: FRA -> ENG"):
        en_results.append(out['translation_text'])

    print("Reassembling fragments...")
    paraphrased_dict = {}
    for i, original_idx in enumerate(chunk_map):
        if original_idx not in paraphrased_dict:
            paraphrased_dict[original_idx] = []
        paraphrased_dict[original_idx].append(en_results[i])

    final_texts = []
    for idx in remaining_df.index:
        final_texts.append(' '.join(paraphrased_dict[idx]))

    remaining_df['paraphrased_text'] = final_texts
    remaining_df['engine'] = 'NLLB-1.3B-UltraBatch'

    final_df = pd.concat([processed_df, remaining_df], ignore_index=True)
    final_df.to_csv(output_file, index=False)
    print(f"Mission Accomplished! Saved to {output_file}")

if __name__ == "__main__":
    df_input = pd.read_csv(input_file)
    process_ultra(df_input)

## Testing

test the semantic similarity function using `all-MiniLM-L6-v2` model

In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer, util
import torch

df = pd.read_csv("old_data/paraphrasing/nllb/DB_test_paraphrased_1.3B_ULTRA.csv")

model = SentenceTransformer('all-MiniLM-L6-v2')

def calculate_semantic_similarity(original_list, paraphrased_list):
    embeddings1 = model.encode(original_list, convert_to_tensor=True)
    embeddings2 = model.encode(paraphrased_list, convert_to_tensor=True)
    
    cosine_scores = util.cos_sim(embeddings1, embeddings2)
    return torch.diagonal(cosine_scores).tolist()

print("Calculating semantic similarity...")
df['semantic_similarity'] = calculate_semantic_similarity(df['Text'].tolist(), df['paraphrased_text'].tolist())

print(df['semantic_similarity'].describe())


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.histplot(df['semantic_similarity'], bins=50, kde=True, color='teal')

plt.title('Distribution of Semantic Similarity')
plt.xlabel('Similarity Score')
plt.ylabel('Frequency')
plt.grid(axis='y', alpha=0.3)

plt.axvline(df['semantic_similarity'].mean(), color='red', linestyle='--', label=f'Mean: {df["semantic_similarity"].mean():.2f}')
plt.legend()

plt.show()

In [ ]:
# Drop low semantic similarity samples

df = df[df['semantic_similarity'] >= 0.60].copy()
print(f"Cleaned! Kept {len(df)} high-quality paraphrased samples.")

mistral=df[df["Writer"]=="mistral-7B"]["Text"]
gemma=df[df["Writer"]=="gemma-2-9b"]["Text"]
qwen=df[df["Writer"]=="qwen-2-72B"]["Text"]
llama=df[df["Writer"]=="llama-8B"]["Text"]
yi=df[df["Writer"]=="accounts/yi-01-ai/models/yi-large"]["Text"]
gpt4o=df[df["Writer"]=="GPT_4-o"]["Text"]
human=df[df["Writer"]=="Human"]["Text"]

df_final=pd.DataFrame({
    "mistral-7B_translated":mistral.reset_index(drop=True),
    "gemma-2-9b_translated":gemma.reset_index(drop=True),
    "qwen-2-72B_translated":qwen.reset_index(drop=True),
    "llama-8B_translated":llama.reset_index(drop=True),
    "yi-large_translated":yi.reset_index(drop=True),
    "GPT_4-o_translated":gpt4o.reset_index(drop=True),
    "Human_translated":human.reset_index(drop=True),
})
df_final.to_csv("old_data/paraphrasing/nllb/DB_test_paraphrased_1.3B_FINAL.csv",index=False)

# Gemini

In [ ]:
df=pd.read_csv("../data/articles.csv")

In [ ]:
limited_df = df.copy()

In [ ]:
limited_df=limited_df.groupby("Writer").sample(n=300, random_state=999)

In [ ]:
limited_df.to_csv("../data/limited_articles.csv", index=False)

In [ ]:
import pandas as pd
import google.generativeai as genai
import time
from tqdm import tqdm
import os
from dotenv import load_dotenv

load_dotenv() 
API_KEY = os.environ.get('GEMINI_API_KEY')

if not API_KEY:
    raise ValueError("API_KEY environment variable not set.")
genai.configure(api_key=API_KEY)
MODEL_NAME = 'gemma-3-27b-it' 
model = genai.GenerativeModel(MODEL_NAME)

INPUT_FILE = "../data/limited_articles.csv"
OUTPUT_FILE = "../data/paraphrased_articles.csv"
NUM_SAMPLES = 2100

def paraphrase_text(text):
    prompt = f"""
    Act as a New York Times writer. Paraphrase the following text while keeping 
    the core meaning intact. Use natural, slightly varied sentence structures 
    to make it look human-written. Do not add any introductory remarks.
    
    Original Text: {text}
    """
    try:
        response = model.generate_content(prompt)
        if response and response.text:
            return response.text.strip()
    except Exception as e:
        print(f"\n[!] Error: {e}")
        return None

df = pd.read_csv(INPUT_FILE)
random_seed = 999

if os.path.exists(OUTPUT_FILE):
    df_progress = pd.read_csv(OUTPUT_FILE)
    processed_count = len(df_progress)
    print(f" continue from : {processed_count}")
else:
    df_progress = pd.DataFrame(columns=["Writer","Original_Text","Paraphrased_Text"])
    processed_count = 0

for index, row in tqdm(df.iterrows(), total=NUM_SAMPLES):
    if index < processed_count:
        continue
        
    original_text = row['Article']
    result = paraphrase_text(original_text)
    
    if result:
        new_row = pd.DataFrame({
            "Writer": [row['Writer'] + "_paraphrased"],
            "Original_Text": [original_text],
            "Paraphrased_Text": [result]
        })
        new_row.to_csv(OUTPUT_FILE, mode='a', index=False, header=not os.path.exists(OUTPUT_FILE))
    else:
        print(f"{index} - Quota error encountered. Waiting before retrying...")
        time.sleep(30)
    time.sleep(4)

In [ ]:
import pandas as pd

df=pd.read_csv('../data/paraphrased_articles.csv')
df.rename(columns={'Writer': 'Writer', 'Article': 'Paraphrased_Text'}, inplace=True)

df.to_csv('../data/paraphrased_articles.csv', index=False)